<a href="https://colab.research.google.com/github/Koroush-Mashayekh/diabetes-risk-prediction/blob/main/02_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

URL = "https://raw.githubusercontent.com/Koroush-Mashayekh/diabetes-risk-prediction/main/data/diabetes.csv"
df = pd.read_csv(URL)

print(df.shape)
df.head()

(768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 1. Train/test split
80% of the data is used for training and 20% is held out as a final test set.
The split is **stratified** to keep the same class ratio (65/35) in both sets,
and `random_state=42` makes the split reproducible.

**Result:** 614 training samples and 154 test samples.
The diabetes ratio is almost identical in both sets (34.9% vs. 35.1%), confirming the stratified split.
The test set will only be used for the final evaluation.

In [2]:
from sklearn.model_selection import train_test_split

X= df.drop(columns = "Outcome")
y= df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = 42)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Diabetes ratio in train:", round(y_train.mean(), 3))
print("Diabetes ratio in test:", round(y_test.mean(), 3))

Train: (614, 8) Test: (154, 8)
Diabetes ratio in train: 0.349
Diabetes ratio in test: 0.351


## 2. Baseline model
A naive model that always predicts the most frequent class ("no diabetes").
Any useful model must clearly outperform this baseline.

**Result:** The baseline reaches **64.9% accuracy** but a **recall of 0.0**: it does not detect a single
diabetic patient. Its ROC-AUC of 0.5 corresponds to random guessing.

This confirms that accuracy alone is misleading on this dataset.
Any useful model must achieve a ROC-AUC clearly above 0.5 and a meaningful recall.

In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

baseline = DummyClassifier(strategy = "most_frequent")
baseline.fit(X_train, y_train)

y_pred = baseline.predict(X_test)
y_proba = baseline.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Recall:", round(recall_score(y_test, y_pred), 3))
print("ROC_AUC:", round(roc_auc_score(y_test, y_pred), 3))

Accuracy: 0.649
Recall: 0.0
ROC_AUC: 0.5
